# Phase 12: End-to-End Pipeline Integration Validation

This notebook imports the actual **FastAPI backend validation services** and runs mock test requests locally to ensure the entire AI pipeline works end-to-end.

It tests various edge-case samples (selfies, blurry shots, multiple leaves, single healthy/diseased leaves) and trace outputs.

In [7]:
# Setup paths and imports (including backend folder to sys.path)
from pathlib import Path
import sys
import os
import torch

PROJECT_ROOT = Path.cwd()
while PROJECT_ROOT.name != 'ai' and PROJECT_ROOT.parent != PROJECT_ROOT:
    if (PROJECT_ROOT / 'ai').exists():
        PROJECT_ROOT = PROJECT_ROOT / 'ai'
        break
    PROJECT_ROOT = PROJECT_ROOT.parent

BACKEND_ROOT = PROJECT_ROOT.parent / "backend"
if str(BACKEND_ROOT) not in sys.path:
    sys.path.insert(0, str(BACKEND_ROOT))

# Import actual production gateway classes
from services.ai_gateway import AIGateway
from services.image_validator import ImageQualityValidator
from services.leaf_detector import YoloLeafDetector
from services.disease_classifier import PytorchDiseaseClassifier

print("✅ Production backend pipeline modules loaded successfully!")

✅ Production backend pipeline modules loaded successfully!


In [8]:
# Initialize production pipeline
MODEL_DIR = PROJECT_ROOT / "models"
LEAF_DETECTOR_MODEL = MODEL_DIR / "detector" / "leaf_detector.pt"

# Initialize stages
quality_validator = ImageQualityValidator()
leaf_detector = YoloLeafDetector(model_path=str(LEAF_DETECTOR_MODEL))

# Wrap TorchScript Crop Classifier
class MockClassifier:
    async def classify(self, image_bytes):
        from services.ai_gateway import AIStageResponse
        return AIStageResponse(
            success=True, 
            confidence=0.96, 
            message="Mock Crop prediction completed.",
            metadata={"class": "Tomato___Late_blight", "health_score": 0.25}
        )

classifier = MockClassifier()
gateway = AIGateway(quality_validator, leaf_detector, classifier)
print("✅ AI Gateway integration pipeline successfully initialized.")

[INFO] Loaded COCO YOLOv8n object filter from o:\Hackthons\KrishiOS\ai\models\detector\yolov8n.pt
[INFO] Loaded YOLO Leaf Detector from o:\Hackthons\KrishiOS\ai\models\detector\leaf_detector.pt
✅ AI Gateway integration pipeline successfully initialized.


### Run Test Pipeline Validation Traces

We can pass standard image files (or mock byte buffers) through the gateway to confirm standard failure/success reporting.

In [9]:
import asyncio

# Create a mock 10x10 empty black image byte array to simulate quality validation failure
from PIL import Image
import io

mock_img = Image.new('RGB', (100, 100), color='black')
byte_arr = io.BytesIO()
mock_img.save(byte_arr, format='JPEG')
mock_bytes = byte_arr.getvalue()

async def test_pipeline():
    print("Running validation check on mock dark/blurry image...")
    res = await gateway.process_image(mock_bytes)
    print("Result Success :", res.get("success"))
    print("Result Stage   :", res.get("stage"))
    print("Result Message :", res.get("message"))
    print("Result Metadata:", res.get("metadata"))

# Run in Jupyter event loop
await test_pipeline()

Running validation check on mock dark/blurry image...
Result Success : False
Result Stage   : quality_validation
Result Message : Improve lighting (image is too dark).
Result Metadata: {'brightness': 0.0}
